In [1]:
import pandas as pd

Link To Dataset

https://www.kaggle.com/datasets/notshrirang/spotify-million-song-dataset

In [2]:
df = pd.read_csv("spotify_millsongdata.csv")

In [3]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [4]:
df.tail(5)

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [5]:
df.shape

(57650, 4)

In [6]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [7]:
df =df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [8]:
df.head(10)

,artist,song,text
0,Keith Green,Don't You Wish You Had The Answers,"Look out your window, see the earth. \r\nWher..."
1,Bon Jovi,"All Talk, No Action","First time I saw you girl, \r\nI knew that it..."
2,Kylie Minogue,After Dark,"That's right, \r\nLet me give you something f..."
3,Rick Astley,What The World Needs Now,"What the world needs now, \r\nIs love, sweet ..."
4,Squeeze,Daphne,Daphne \r\nDon't be ridiculous \r\nThis sile...
5,Jimmy Buffett,Barefoot Children,Scratch my back with a lightning bolt \r\nThu...
6,Sublime,Roots Of Creation,"One two three four! \r\nPull up here honey, i..."
7,O.A.R.,So Moved On,"I woke today, felt another way, \r\nEverythin..."
8,Eddie Cochran,Pocketful Of Hearts,"A pocketful of hearts, a-baby you collect 'em ..."
9,Reba Mcentire,A Little Want To,There ain't no excuse that's what my mama said...


In [9]:
df['text'][0]

"Look out your window, see the earth.  \r\nWhere did it come from, who gave it birth.  \r\nWhere did it come from, where will it go, where will it go?  \r\nDon't you wish you had the answers, well, I know.  \r\nSee how the rain falls, who made the sky?  \r\nIt's never ending, and you wonder why.  \r\nWhere did it come from, where will it go, where will it go?  \r\nDon't you wish you had the answers, well I know.  \r\nJust look out past the stars, look to the one who put them there.  \r\nHe, he made them all, and he gave them to us to share.  \r\nYes he made them all, and he's gonna take them all back someday.  \r\nDon't you just wonder, what lies ahead?  \r\nThere's peace in knowing, what jesus said.  \r\nWhere did you come from? where will you go, where will you go?  \r\nDon't you wish you had the answers?  \r\nDon't you wish you had the answers?  \r\nDon't you wish you had the answers?  \r\nWell, I know.\r\n\r\n"

In [10]:
df.shape

(5000, 3)

Text Cleaning/ Text Preprocessing

In [11]:
df['text'] = df['text'].str.lower().replace(r'^\w\s', ' ').replace(r'\n', ' ', regex = True)

In [12]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [13]:
df['text'] = df['text'].apply(lambda x: tokenization(x))

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)

In [16]:
similarity[0]

array([1.        , 0.02568198, 0.01748876, ..., 0.07034984, 0.05104717,
       0.08976162], shape=(5000,))

In [25]:
df[df['song'] == "Barefoot Children"][['song','artist']].values[0]

array(['Barefoot Children', 'Jimmy Buffett'], dtype=object)

In [26]:
def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similarity[idx])),reverse=True,key=lambda x:x[1])
    
    songs = []
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)
        
    return songs

In [28]:
recommendation('Barefoot Children')

['Crying In The Rain',
 'Crush',
 'Just The Rain',
 'Children Of The World',
 'Dreams',
 'New Machine',
 "I'm Sick Y'all",
 'Blue Hotel',
 'Hatfield',
 "For What It's Worth",
 "Everybody's Sweetheart",
 'Singing In The Rain',
 'Ball And Chain',
 "It's Raining Men",
 'Beautiful',
 "She's My Kind Of Rain",
 'Cloudburst',
 'If You Love',
 'Stormy Weather',
 'Barefoot Floors']

In [29]:
import pickle
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))